In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support

sys.path.append(os.getcwd())

from feature_pipeline import prepare_datasets
from transformer_model import (
    TrainingConfig,
    build_user_sequences,
    make_dataloaders,
    ChurnTransformer,
    train_model,
    predict_proba,
    UserSequenceDataset,
    collate_batch,
)


In [ ]:
BASE_PATH = os.getcwd()
train_path = f'{BASE_PATH}/churn-prediction-25-26/train.parquet'
test_path = f'{BASE_PATH}/churn-prediction-25-26/test.parquet'

config = TrainingConfig(
    d_model=256,
    nhead=4,
    num_layers=4,            
    dim_feedforward=768,    
    dropout=0.15,           

    max_seq_len=450,       
    batch_size=256,
    num_workers=4,

    lr=1e-5,                 
    weight_decay=1.5e-3,     
    epochs=60,

    use_cosine_decay=True,
    eta_min_factor=0.05,
    warmup_epochs=3,
    use_focal_loss=True,
    focal_gamma=1.6,
    
    early_stop_patience=5,
    early_stop_min_delta=1e-4,
)

# Keep prepare_datasets parameters unchanged

train_df, val_df, test_df, labels, artifacts = prepare_datasets(
    train_path,
    test_path,
    val_ratio=0.2,
    random_state=42,
    truncate_buffer_min=2,
    truncate_buffer_frac=0.02,
    cutoff_time='2018-11-09',
    # cutoff_time=11,
    # drop_inactive_before_cutoff=False,
)
label_dict = labels.to_dict()
print('train rows', len(train_df), 'val rows', len(val_df), 'test rows', len(test_df))


train rows 11086357 val rows 2741890 test rows 4393179


: 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_sequences = build_user_sequences(train_df, artifacts.numeric_cols, config.max_seq_len)

val_sequences = build_user_sequences(val_df, artifacts.numeric_cols, config.max_seq_len)

train_loader, val_loader = make_dataloaders(train_sequences, val_sequences, label_dict, config)


# The dataset is imbalanced; use pos_weight plus focal loss
pos_weight = max(1.0, float((labels == 0).sum() / (labels == 1).sum()))
print('pos_weight set to', pos_weight)


model = ChurnTransformer(
    num_numeric=len(artifacts.numeric_cols),
    num_pages=len(artifacts.page_categories),
    num_metro=len(artifacts.metro_mapping),
    num_state=len(artifacts.state_mapping),
    num_device=len(artifacts.device_mapping),
    config=config,
).to(device)
print(model)


In [ ]:
train_losses, val_losses, model, best_epoch, best_state = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=config.epochs,
    lr=config.lr,
    weight_decay=config.weight_decay,
    pos_weight=pos_weight,
    use_cosine_decay=config.use_cosine_decay,
    eta_min_factor=config.eta_min_factor,
    warmup_epochs=config.warmup_epochs,
    use_focal_loss=config.use_focal_loss,
    focal_gamma=config.focal_gamma,
    early_stop_patience=config.early_stop_patience,
    early_stop_min_delta=config.early_stop_min_delta,
)
print('done training, best epoch:', best_epoch)


In [ ]:
# Save the best validation model and artifacts
ckpt = {
    'model_state': best_state,
    'config': config.__dict__,
    'artifacts': artifacts,
    'numeric_cols': artifacts.numeric_cols,
    'best_epoch': best_epoch,
}
torch.save(ckpt, 'transformer_best.pt')
print('saved to transformer_best.pt (best epoch)', best_epoch)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()


In [ ]:
# Evaluate on validation set
val_probs = predict_proba(model, val_loader, device)
val_index = list(val_sequences.keys())
val_true = labels.loc[val_index]
val_pred = pd.Series(val_probs).reindex(val_index)

thresholds = np.round(np.arange(0.10, 0.901, 0.02), 2)
threshold_metrics = []
for thr in thresholds:
    preds = (val_pred >= thr).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(val_true, preds)
    threshold_metrics.append(
        {
            'threshold': float(thr),
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'accuracy': acc,
        }
    )

threshold_df = pd.DataFrame(threshold_metrics).sort_values('threshold')
best_row = max(threshold_metrics, key=lambda x: x['f1'])
best_threshold = best_row['threshold']
best_f1 = best_row['f1']
val_acc = best_row['accuracy']
val_pred_binary = (val_pred >= best_threshold).astype(int)
auc = roc_auc_score(val_true, val_pred)
print(f"Best F1={best_f1:.4f} at threshold={best_threshold:.2f}")
print(
    "Validation accuracy@best: {acc:.4f}, precision={prec:.4f}, recall={rec:.4f}".format(
        acc=best_row['accuracy'], prec=best_row['precision'], rec=best_row['recall']
    )
)
print(f"Validation ROC-AUC: {auc:.4f}")
threshold_df.sort_values('f1', ascending=False).head()


In [ ]:

from pathlib import Path
from datetime import datetime
import csv


def _prev_best_score(log_path: Path):
    if not log_path.exists():
        return None
    best = None
    with log_path.open() as f:
        for row in csv.DictReader(f):
            for key in ('kaggle_score', 'val_auc', 'val_acc'):
                val = row.get(key)
                try:
                    val_f = float(val)
                except Exception:
                    continue
                if best is None or val_f > best:
                    best = val_f
    return best


def _next_action(train_losses, val_losses, best_row, config, kaggle_score):
    suggestions = []
    if train_losses and val_losses:
        gap = val_losses[-1] - train_losses[-1]
        if gap > 0.05:
            suggestions.append('Mild overfitting: raise dropout to 0.18-0.22 or weight_decay to 2e-3; compare max_seq_len=400')
        elif val_losses[-1] > 0.45:
            suggestions.append('Some underfitting: num_layers=5 or dim_feedforward=896; run +3-5 epochs (warmup=4)')
    if best_row['recall'] + 0.1 < best_row['precision']:
        suggestions.append('Recall too low: pos_weight *1.2 or focal_gamma +0.2; densify threshold grid 0.35-0.55')
    if best_row['precision'] + 0.1 < best_row['recall']:
        suggestions.append('Precision too low: pos_weight *0.8 or focal_gamma -0.2; shift threshold up to 0.55-0.65')
    if kaggle_score and kaggle_score > 0.65:
        suggestions.append('Near 0.67: small scan around current config (lr=5e-4/7e-4, max_seq_len=400/500)')
    if not suggestions:
        suggestions.append('Decide after inspecting loss curves: if smooth, increase width/layers and slightly tune lr')
    return '; '.join(suggestions[:3])


test_sequences = build_user_sequences(test_df, artifacts.numeric_cols, config.max_seq_len)

test_dataset = UserSequenceDataset(test_sequences, labels=None, user_ids=list(test_sequences.keys()))

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    collate_fn=collate_batch,
)

test_probs = predict_proba(model, test_loader, device)
test_pred = pd.Series(test_probs)

submission = pd.DataFrame({
    'id': list(test_pred.index),
    'target': (test_pred >= best_threshold).astype(int).values
})
submission['id'] = pd.to_numeric(submission['id'], errors='ignore')
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')

# Submit to Kaggle and log results (requires ~/.kaggle/kaggle.json)
SUBMIT_TO_KAGGLE = True
submission_info = None
run_note = f"transformer epoch{best_epoch} thr={best_threshold:.2f}"

if SUBMIT_TO_KAGGLE:
    try:
        from kaggle_submit import submit_and_track

        submission_info = submit_and_track(
            'submission.csv',
            competition='churn-prediction-25-26',
            message=run_note,
            wait_for_result=True,         # set False to upload without waiting
            poll_interval=30,
            max_polls=8,
            log_path='submission_log.csv',
            extra_meta={
                'best_epoch': int(best_epoch),
                'best_threshold': float(best_threshold),
                'val_auc': float(auc),
                'val_f1': float(best_f1),
                'val_acc': float(val_acc),
            },
        )
        print('Submission done:', submission_info)
    except Exception as e:
        print('Kaggle submit skipped:', e)

log_path = Path('tuning_log.csv')
prev_best = _prev_best_score(log_path)
kaggle_score = submission_info.get('score') if submission_info else None
new_score = kaggle_score if kaggle_score is not None else float(auc)
improved = prev_best is None or (new_score is not None and prev_best is not None and new_score > prev_best)

next_action = _next_action(train_losses, val_losses, best_row, config, kaggle_score)

log_row = {
    'run_time_utc': datetime.utcnow().isoformat(timespec='seconds'),
    'run_note': run_note,
    'best_epoch': int(best_epoch),
    'best_threshold': float(best_threshold),
    'val_auc': float(auc),
    'val_f1': float(best_f1),
    'val_acc': float(val_acc),
    'train_loss_last': float(train_losses[-1]),
    'val_loss_last': float(val_losses[-1]),
    'val_loss_min': float(min(val_losses)),
    'kaggle_score': kaggle_score,
    'kaggle_status': submission_info.get('status') if submission_info else None,
    'kaggle_ref': submission_info.get('ref') if submission_info else None,
    'improved_vs_prev': improved,
    'next_action': next_action,
    'd_model': config.d_model,
    'nhead': config.nhead,
    'num_layers': config.num_layers,
    'dim_feedforward': config.dim_feedforward,
    'dropout': config.dropout,
    'max_seq_len': config.max_seq_len,
    'batch_size': config.batch_size,
    'num_workers': config.num_workers,
    'lr': config.lr,
    'weight_decay': config.weight_decay,
    'epochs': config.epochs,
    'use_cosine_decay': config.use_cosine_decay,
    'eta_min_factor': config.eta_min_factor,
    'warmup_epochs': config.warmup_epochs,
    'use_focal_loss': config.use_focal_loss,
    'focal_gamma': config.focal_gamma,
    'pos_weight': float(pos_weight),
}
field_order = [
    'run_time_utc','run_note','best_epoch','best_threshold',
    'val_auc','val_f1','val_acc','train_loss_last','val_loss_last','val_loss_min',
    'kaggle_score','kaggle_status','kaggle_ref','improved_vs_prev','next_action',
    'd_model','nhead','num_layers','dim_feedforward','dropout','max_seq_len',
    'batch_size','num_workers','lr','weight_decay','epochs','use_cosine_decay',
    'eta_min_factor','warmup_epochs','use_focal_loss','focal_gamma','pos_weight'
]
needs_header = not log_path.exists()
with log_path.open('a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=field_order)
    if needs_header:
        writer.writeheader()
    writer.writerow(log_row)
print('Logged to tuning_log.csv (improved:', improved, ')')

submission.head()
